In [2]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=ffQ4w0n6DMZaymvzIkZwzguri4CjX3&access_type=offline&code_challenge=Mwit1p3ZvhA_WiEwiRSZTjLT00Uwp3c46kMvQw3FdUA&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [3]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [4]:
def build_prompts():
    return {
        "CODE_LOOKUP": """PART 1: PROPERTY IDENTIFICATION

| Field                     | Value                         | Source Citation                      |
|---------------------------|-------------------------------|--------------------------------------|
| Full Street Address       | 8705 COUNTY ROAD 206A, ALVARADO, TX 76009 | Document page 1, Property section    |
| Municipality/Jurisdiction | Alvarado                      | Document page 1, Property section    |
| County                    | Johnson County (determined from ZIP 76009) | Address verification                 |
| ZIP Code                  | 76009                         | Document page 1, Property section    |
| Inspection/Report Date    | 6/25/2024                     | Document page 1, Date Inspected      |
| Carrier Estimate Date     | 2/12/2025                     | Document page 2, Estimate footer     |
| Initial Carrier Total     | $16,113.56                    | Document page 7, Replacement Cost Value |
| Price List Code           | TXDF8X_FEB25                  | Document page 1, Price List          |

TAX RATE DETERMINATION:
- ZIP Code: 76009
- County: Johnson County
- Municipality: Alvarado
        CODE STACK DETERMINATION
 
Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection:
 
| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | [Year] IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | [Year] NEC | [NEC.gov] | [none] | Mandatory |
| IECC | [Year] IECC | [DOE/state site] | [state energy code mods] | Mandatory |
 
For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

Also get the tax rates for that specific zip codes.
""",
 
        "REPORT_ANALYSIS": """You are a forensic damage analyst and good-faith adjuster specialist focusing on residential property insurance claims. You must follow standardized measurement and documentation protocols plus contradiction detection to ensure consistent analysis across multiple runs.

OBJECTIVE:
Perform complete, evidence-based analysis of all observable damages using standardized measurement hierarchy, documentation requirements, and systematic contradiction detection.

MEASUREMENT SOURCE PRIORITY (Use Highest Available - NO EXCEPTIONS):

HIERARCHY:
1. PROFESSIONAL MEASUREMENT REPORTS (Highest Reliability)
   - EagleView, Pictometry, aerial measurement data
   - Licensed surveyor measurements
   - Engineering inspection dimensions with field verification

2. INSPECTION DOCUMENTATION (High Reliability)
   - Written measurements in forensic reports
   - Engineer's field notes with specific dimensions
   - Adjuster measurements with photo verification

3. PHOTO ANALYSIS WITH SCALING (Medium Reliability)
   - Use known references (doors = 7', standard brick = 3", etc.)
   - Document scaling method and reference points
   - Cross-verify with multiple photos when possible

4. CARRIER ESTIMATES (Validation Only - Lowest Priority)
   - Use ONLY to validate measurements from higher sources
   - Never as primary measurement source
   - Challenge significant variances with documented evidence

MEASUREMENT DOCUMENTATION REQUIREMENTS:
For every quantity, document:
- Quantity: [Number with decimals]
- Unit: [SF/LF/EA/SQ/etc.]
- Source: [Specific report page or photo ID]
- Method: [Direct measurement/scaling/calculation]
- Confidence: [High/Medium/Low]
- Cross-Reference: [Verification source if available]

PART 1: PHOTO CAPTION CONTRADICTION ANALYSIS (NEW)

Systematically compare photo captions against denial letters and carrier communications:

CONTRADICTION DETECTION PROTOCOL:
- Extract all photo captions from inspection files
- Cross-reference against denial letter statements
- Flag contradictions between observed conditions and carrier conclusions
- Document evidence suppression or mischaracterization

| Photo ID | Caption Text | Carrier Statement | Contradiction Type | Evidence Impact |
|----------|--------------|-------------------|-------------------|-----------------|
| IMG_001 | "Significant hail damage to shingles" | "No storm damage observed" | Direct contradiction | Establishes causation |

PART 2: ENGINEERING REPORT CONTRADICTION DETECTION (NEW)

Flag conflicts between engineering conclusions and documented evidence:

ENGINEERING ANALYSIS:
- Compare engineer's conclusions to photographic evidence
- Identify contradictions between field notes and final report
- Flag omissions of documented damage in conclusions
- Cross-reference measurements between different sections

| Report Section | Engineer Conclusion | Contradicting Evidence | Evidence Source | Impact Assessment |
|----------------|-------------------|----------------------|-----------------|-------------------|
| Roof Analysis | "No structural damage" | Photo shows cracked decking | IMG_045 | Understated damage |

ANALYSIS STRUCTURE:
Repeat the following format for every room, elevation, or system area with documented or inferable damage.

### [Room or Elevation Name]

DAMAGE DOCUMENTATION:
- Primary Damage: Describe the main damage (e.g., water stain, blistering, rot, delamination)
- Secondary Damage: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
- Evidence Sources: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes

AFFECTED COMPONENTS:
List all building components that show damage or require restoration work:
- Structural Elements: (e.g., ceiling joists, wall framing, roof decking)
- Finish Materials: (e.g., drywall, paint, flooring, trim)
- Systems: (e.g., electrical fixtures, HVAC components, plumbing)
- Insulation/Barriers: (e.g., insulation, vapor barriers, house wrap)

MEASUREMENT EXTRACTION:
- Damaged Area Dimensions: [Length × Width × Height with source]
- Affected Component Quantities: [Number of units with measurement source]
- System Impacts: [Linear feet, square feet, etc. with confidence level]

CARRIER ESTIMATE COMPARISON:
- Included Scope: List what the carrier did include (line item description)
- Missing Scope: Items observed but omitted in carrier scope
- Quantity Variances: Compare carrier measurements to documented evidence

CONTRADICTION SUMMARY:
- Photo vs. Carrier Statement Conflicts: [List all identified contradictions]
- Engineering vs. Evidence Conflicts: [List all report contradictions]
- Internal Document Inconsistencies: [Flag self-contradictions]

EVIDENCE CORRELATION GUIDELINES:
For every damage condition, you must:
- Link to at least one photo ID or annotated image
- Cite page number or section from relevant inspection or engineering report
- If damage extent is inferred, specify the method used
- Do not make undocumented assumptions; flag any gaps explicitly

VALIDATION PROTOCOLS:
- Every damage claim must have at least one evidence reference
- All measurements must be traceable to documented sources
- Cross-reference all measurement sources when available
- Flag estimates vs. exact measurements clearly
- Document all contradictions with specific evidence

MANDATORY OUTPUT REQUIREMENTS:
1. Complete one section per distinct room/elevation/system
2. Document all measurement sources using hierarchy
3. Tie every observed damage to verifiable evidence
4. Include confidence levels for all measurements
5. Cross-reference carrier estimate quantities where applicable
6. Provide complete contradiction analysis with evidence

Complete the output for all rooms or elevations with observed damage using standardized measurement protocols and contradiction detection.
"""
,
 
        "DAUBERT_ESTIMATE_OUTPUT": """
You are a forensic Daubert-compliant expert witness preparing a final "Plaintiff-style" Xactimate estimate report ready for legal submission.
 
OBJECTIVES:
1. Synthesize all upstream analyses into one cohesive document.
2. Mirror the layout, level of detail, and citation style in the Marc Arnold Estimate (Grace Forensic Loss Consultants, April 2025):
   - Cover page with case header (Insured, Claim #, Property, Dates)
   - Table of Contents
   - Line-item tables by area (roof, exterior elevations, general conditions, etc.) in Xactimate format with CAT/SEL codes
   - Summaries (by elevation, by category, grand totals) with precise math
   - "Daubert Reliability" section noting sources, known error rates, peer-review references, and confirmation of code/version accuracy
   - Appendices for photos, code citation library, and evidence matrix
 
REQUIREMENTS:
- Use exact Xactimate table columns:
 
| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |
 
- Include a formal "Expert Opinion & Methodology" narrative conforming to Daubert standards
- All citations must reference either your prior stage ("[Stage] Output") or the Marc Arnold PDF (e.g. "Grace Forensic Loss Consultants, p. 2")
- Maintain legal-grade formality and ready-for-court structure
 
FINAL OUTPUT:
Produce a single Markdown (or PDF-ready) document that a court could receive as the expert's estimate exhibit.
""",
 
        "SCOPING_LOGIC": """You are a restoration estimator and on-the-ground contractor specialist building a complete plaintiff-style scope justification using standardized sequential methodology with automated safety and logistics protocols to ensure consistent scope determination across multiple runs.
OBJECTIVE: 
Generate complete scope justification using unified four-step methodology with automated additions based on scope thresholds. All measurements will be handled in the estimation stage.
UNIFIED SCOPE METHODOLOGY:
Apply this four-step hierarchy in exact sequence - NO SKIPPING OR REORDERING:
STEP 1: PHYSICAL DAMAGE REQUIREMENTS (Primary Driver)
- Replace all components with documented damage
- Base scope on photographic evidence and inspection findings
- Include only items with clear damage documentation
- Example: Cracked shingles → Replace damaged shingles
STEP 2: CODE-TRIGGERED REQUIREMENTS (Secondary)
- Items required when physical work disturbs building assemblies
- Triggered only by work from Step 1
- Must cite specific code sections and trigger conditions
- Example: Roof tear-off → Triggers decking inspection per IRC R908.3.1
STEP 3: INDUSTRY STANDARD PRACTICES (Tertiary)
- Work unavoidable due to construction sequence
- Items that cannot be reused once disturbed
- Components with single-use specifications
- Example: Pipe jacks → Must replace when roof is replaced (single-use items)
STEP 4: AESTHETIC/MATCHING REQUIREMENTS (Final)
- When partial repair creates visible mismatch
- Material discontinuation or availability issues
- Line-of-sight uniformity requirements
- Example: One damaged siding panel → Replace full elevation for color match
AUTOMATED LOGISTICS ADDITION (NEW):
Based on scope analysis, automatically add required logistics:
THRESHOLD-BASED LOGISTICS:
- 3+ Trade Coordination: Auto-add project manager (40 hrs minimum)
- Roofing Work >10 SQ: Auto-add dumpster (30-yard minimum)  
- Interior Work >500 SF: Auto-add temporary facilities
- >$50K Total Scope: Auto-add daily cleanup and protection
- Multi-story Work: Auto-add scaffolding or lift rental
AUTOMATED OSHA SAFETY REQUIREMENTS (NEW):
Flag mandatory safety requirements based on work type:
SAFETY PROTOCOL AUTOMATION:
- Fall Protection: Required for work >6 feet (roofing, multi-story)
- Respiratory Protection: Required for mold/asbestos/dust work
- Electrical Safety: LOTO procedures for electrical work
- Confined Space: Entry procedures for crawlspaces/basements
- Hazmat Protocols: Lead/asbestos testing and containment
| Work Type | OSHA Standard | Auto-Triggered Items | Cost Impact |
|-----------|---------------|---------------------|-------------|
| Roofing | 29 CFR 1926.501 | Fall protection, safety harnesses | Add to estimate |
THIRD-PARTY AUTHORITY INTEGRATION (NEW):
Automatically check scope against industry standards:
AUTHORITY DATABASE QUERIES:
- HAAG Standards: Hail damage assessment protocols
- NRCA Guidelines: Roofing installation standards  
- IICRC Standards: Water damage restoration protocols
- FEMA Guidelines: Storm damage assessment standards
| Scope Item | Authority Standard | Compliance Requirement | Auto-Added Scope |
|------------|-------------------|----------------------|------------------|
| Water damage restoration | IICRC S500 | Moisture monitoring, antimicrobial treatment | 72-hr monitoring protocol |
SCOPE CATEGORIES:
1. ROOFING SYSTEM REQUIREMENTS:
For each component, include:
- Code Requirement (w/ citation)
- What triggers the requirement (e.g., tear-off, disturbed assembly)
- Required Work Items (demo + install, inspection, testing)
- Consequences of omission (warranty void, leaks, code violation)
Standard inclusions:
- Decking inspection (when tear-off occurs)
- Underlayment replacement (code requirement)
- Ventilation compliance (when system is opened)
- Flashing replacement (with roof replacement)
- High-wind installation (jurisdiction-specific)
2. EXTERIOR REQUIREMENTS:
- Weather barriers (when siding removed)
- Flashing over trim (code requirement)
- Window installation details (proper sealing)
- Matching considerations (elevation continuity)
3. INTERIOR REQUIREMENTS:
- Moisture protocols (IICRC S500 compliance - AUTO-TRIGGERED)
- Insulation replacement (when wet or disturbed)
- Paint coverage (corner-to-corner standards)
- Texture matching (when repairs made)
4. MANDATORY SCOPE INCLUSIONS:
These items are required based on scope sequencing, not visible damage.
Process:
1. Map restoration sequence (demo → rough → finish)
2. Identify unavoidable impacts (what gets disturbed)
3. Specify replacement requirements (what can't be reused)
4. Document industry standards (manufacturer specs, trade practices)
5. Auto-add logistics based on thresholds
6. Auto-add safety requirements based on OSHA standards
Justification format:
- Item: [specific scope component]
- Trigger: [why it's unavoidable or auto-triggered]
- Standard: [industry/manufacturer/OSHA requirement]
- Step Applied: [Which of 4 methodology steps or "AUTO-TRIGGERED"]
5. MATCHING & AESTHETIC REQUIREMENTS:
Explain when partial replacement is inappropriate:
- Define visual mismatch triggers: color, sheen, exposure age
- Apply line-of-sight logic (e.g., hallway ceiling vs. bedroom)
- Detail material availability issues (discontinued trim, aged siding)
- Explain "paint from corner to corner" rule
6. GENERAL CONDITIONS & OVERHEAD (AUTOMATED):
Define what GC-level provisions are triggered based on scope analysis:
- Project manager (AUTO: 3+ trades requirement)
- Dumpster, job toilet, material storage needs (AUTO: based on scope size)
- Permits (when and why needed)
- O&P (AUTO: applied if ≥3 trades OR complex coordination)
- Safety equipment (AUTO: based on OSHA requirements)
VALIDATION PROTOCOL:
- Every scope item must have clear justification
- All code citations must be current and accurate
- Aesthetic standards must be objectively measurable
- Sequence logic must be technically sound
- Each item must reference which methodology step applies or if auto-triggered
- All automated additions must have threshold justification
- NO QUANTITIES OR MEASUREMENTS - scope identification only
FINAL OUTPUT FORMAT:
Organize by the four methodology steps plus automated additions:
STEP 1 - PHYSICAL DAMAGE SCOPE:
[List all items driven by documented damage]
STEP 2 - CODE-TRIGGERED SCOPE:
[List all items required by building codes when Step 1 work occurs]
STEP 3 - INDUSTRY STANDARD SCOPE:
[List all items unavoidable due to construction sequence]
STEP 4 - AESTHETIC/MATCHING SCOPE:
[List all items required for visual continuity]
AUTOMATED LOGISTICS ADDITIONS:
[List all auto-triggered logistics based on thresholds]
AUTOMATED SAFETY REQUIREMENTS:
[List all auto-triggered OSHA safety requirements]
GENERAL CONDITIONS:
[List project management and overhead requirements with auto-trigger justification]
Each item must include:
- Description of work
- Justification/trigger (including auto-trigger thresholds)
- Code citation, industry standard, or OSHA requirement
- Methodology step applied or "AUTO-TRIGGERED"
 
""",
 
        "ESTIMATE": """You are a certified insurance restoration estimator and Xactimate expert creating precise, court-ready cost breakdowns using standardized pricing hierarchy, calculation methodology, and line item relationship verification to ensure consistent estimates across multiple runs.

OBJECTIVE: 
Generate mathematically precise estimates with complete cost calculations, structured source citations, and parent-child line item verification using the unified pricing and calculation protocols.

CRITICAL: This is the ONLY stage that performs quantification. Extract all measurements from the REPORT_ANALYSIS stage output and apply scope from SCOPING_LOGIC stage.

PRICING HIERARCHY (Fixed Priority - NO EXCEPTIONS):

PRICING PROTOCOL:
1. CARRIER ESTIMATE RATES (For Matching Line Items)
   - Use exact unit prices from carrier estimate
   - Apply to identical scope items only
   - Maintain same units and specifications
   - Document as "Source: Carrier estimate line [X]"

2. MARKET RATES (For Missing/Additional Items)
   - Source from regional price lists for carrier estimate date
   - Use price list code from carrier estimate
   - Apply standard grade specifications unless documented otherwise
   - Research rates for month/year of carrier estimate generation
   - Document as "Source: [Regional price list code] - [Month/Year]"

PARENT-CHILD LINE ITEM VERIFICATION (NEW):
Verify line item relationships using Xactimate database logic:

PARENT-CHILD RELATIONSHIP CHECKS:
- Verify parent items include all necessary child components
- Flag missing child items (e.g., removal, disposal, setup)
- Check for double-billing between parent and child items
- Validate quantity relationships between related items

| Parent Item | Required Child Items | Present in Estimate | Missing Components |
|-------------|---------------------|-------------------|-------------------|
| Shingle Installation | Removal, disposal, underlayment | Partial | Missing disposal |

LABOR MINIMUMS VERIFICATION (NEW):
Check specialized trade minimums:

TRADE MINIMUM REQUIREMENTS:
- Electrical: 2-hour minimum for any electrical work
- Plumbing: 2-hour minimum for any plumbing work  
- HVAC: 4-hour minimum for system work
- Specialty Trades: Research current minimums by trade

MEASUREMENT INTEGRATION:
Extract all quantities from REPORT_ANALYSIS stage using the documented measurement hierarchy:
- Use measurements with "High" confidence first
- Cross-reference multiple measurement sources
- Document measurement source for each line item
- Apply quantities to scope items from SCOPING_LOGIC stage

TAX CALCULATION PROTOCOL:
Apply jurisdiction-specific tax rate from CODE_LOOKUP stage:
1. Apply only to material portion of line items
2. For mixed labor/material items: Apply 50/50 split unless specified
3. Labor-only items: $0.00 tax
4. Document rate source from CODE_LOOKUP analysis

CALCULATION REQUIREMENTS:
MANDATORY FORMULAS:
- Direct Cost = Quantity × Unit Price
- Tax = Material Portion × Tax Rate
- O&P = (Direct Cost + Tax) × 0.20
- RCV = Direct Cost + Tax + O&P
- ACV = RCV - Depreciation (typically $0.00)

O&P APPLICATION:
- Apply 10% Overhead + 10% Profit (20% total compounded)
- Required when: 3+ trades OR complex coordination
- Apply to: (Direct Cost + Tax) subtotal
- Document justification for application

SECTION ORGANIZATION:
Use standardized section names:
- "Roofing" (not "Roofing System")
- "Exterior" (not "Exterior Elevations")
- "Interior" (not "Interior Restoration")
- "Electrical" (not "Electrical Systems")
- "General Conditions"

REQUIRED JSON OUTPUT FORMAT:

{
  "property_identification": {
    "address": "From CODE_LOOKUP stage",
    "jurisdiction": "Municipality, County, State",
    "tax_rate": "X.XX% (Source from CODE_LOOKUP)",
    "price_list": "Regional code from carrier estimate",
    "estimate_date": "Carrier estimate generation date",
    "initial_carrier_total": "Final total from carrier estimate"
  },
  "measurement_summary": {
    "source_hierarchy_used": "List primary measurement sources used",
    "confidence_levels": "High/Medium/Low summary for major measurements"
  },
  "pricing_breakdown": {
    "carrier_rate_items": "Count of items using carrier pricing",
    "market_rate_items": "Count of items using regional pricing",
    "pricing_date": "Month/year of rate application"
  },
  "line_item_verification": {
    "parent_child_issues": "Count of missing child items identified",
    "labor_minimum_violations": "Count of trades below minimum hours",
    "double_billing_flags": "Count of potential double-billing issues"
  },
  "sections": [
    {
      "name": "Standard section name from organization rules",
      "line_items": [
        {
          "description": "Detailed scope description from SCOPING_LOGIC",
          "quantity": "From REPORT_ANALYSIS measurement hierarchy",
          "unit": "Standard units (SF/LF/EA/SQ)",
          "unit_price": "From pricing hierarchy with source",
          "tax": "Jurisdiction rate on materials only",
          "o_and_p": "20% of (cost + tax)",
          "rcv": "Total calculated amount",
          "depreciation": "0.00 unless specified",
          "acv": "RCV minus depreciation",
          "measurement_source": "Specific source from REPORT_ANALYSIS",
          "pricing_source": "Carrier rate or market rate with reference",
          "scope_step": "Which methodology step from SCOPING_LOGIC",
          "code_citation": "Supporting code or standard from CODE_LOOKUP",
          "parent_item": "Parent line item if applicable",
          "child_items": "Required child items",
          "labor_minimum_check": "Pass/Fail with minimum hours"
        }
      ],
      "section_total": "Sum of all line items in section"
    }
  ],
  "general_conditions": [
    {
      "description": "From SCOPING_LOGIC general conditions",
      "quantity": "Hours or units",
      "unit": "HR/EA/etc",
      "unit_price": "Market rate with source",
      "total": "Calculated amount",
      "justification": "Requirement from SCOPING_LOGIC",
      "auto_triggered": "Yes/No based on threshold"
    }
  ],
  "totals": {
    "subtotal_all_sections": "Sum of all section totals",
    "general_conditions_total": "Sum of general conditions",
    "grand_total_rcv": "Final estimate amount",
    "variance_from_carrier": "Difference from initial carrier total",
    "variance_percentage": "Percentage difference calculation"
  }
}

INTEGRATION REQUIREMENTS:
- Use exact property data from CODE_LOOKUP stage
- Apply exact measurements from REPORT_ANALYSIS stage
- Follow exact scope from SCOPING_LOGIC stage
- Use exact tax rates from CODE_LOOKUP stage
- Apply pricing hierarchy consistently
- Verify all parent-child line item relationships
- Check all labor minimums

VALIDATION CHECKLIST:
□ All measurements sourced from REPORT_ANALYSIS stage
□ All scope items from SCOPING_LOGIC stage included
□ Carrier rates used for matching line items
□ Market rates used for additional items with date reference
□ Tax calculations use jurisdiction rate from CODE_LOOKUP
□ Section names follow standardization rules
□ All calculations mathematically exact
□ All sources documented with specific references
□ Parent-child relationships verified
□ Labor minimums checked and documented

QUALITY ASSURANCE:
- Cross-reference all stage outputs for consistency
- Verify total calculations multiple times
- Ensure no scope items are omitted
- Confirm all pricing sources are documented
- Validate tax rate application
- Flag all line item relationship issues
- Document all labor minimum violations

This estimate must integrate all prior stage outputs using standardized protocols to ensure consistency across multiple estimate runs.
""",
        "REBUTTAL": """You are a legal strategist and case analyst providing confidence-based findings prioritization using standardized three-tier methodology to ensure consistent tactical guidance across multiple estimate runs.

OBJECTIVE:
Analyze all findings from previous stages and assign confidence levels with specific action recommendations and escalation pathways.

3-TIER CONFIDENCE FRAMEWORK:

TIER 1 - HIGH CONFIDENCE FINDINGS (Automated Win Potential)
Criteria for Tier 1 Classification:
- Clear mathematical errors with documented proof
- Direct code violations with specific citations
- Documented contradictions in carrier materials (photos vs. statements)
- LKQ violations with clear specifications mismatch
- Prompt Payment Act violations with calculated deadlines

TIER 1 ANALYSIS:
For each high-confidence finding:
- Finding Type: [Code violation/Mathematical error/Contradiction/LKQ/PPA]
- Evidence Strength: [Specific documentation/calculations]
- Legal Impact: [Potential violation/penalty amount]
- Recommended Action: [Immediate demand letter/formal complaint]
- Success Probability: 85-95%

TIER 2 - MEDIUM CONFIDENCE FINDINGS (Human Review Required)
Criteria for Tier 2 Classification:
- Industry standard deviations requiring interpretation
- Complex matching/aesthetic disputes with arguable positions
- Engineering report conflicts requiring expert analysis
- Scoping omissions with technical justification needed
- Policy interpretation disputes

TIER 2 ANALYSIS:
For each medium-confidence finding:
- Finding Type: [Industry standard/Matching dispute/Engineering conflict]
- Evidence Strength: [Supporting documentation required]
- Interpretation Complexity: [Technical/Legal factors affecting outcome]
- Recommended Action: [Expert review/additional documentation needed]
- Success Probability: 60-80%

TIER 3 - POTENTIAL FINDINGS (Expert Escalation Required)
Criteria for Tier 3 Classification:
- Hidden damages requiring further investigation
- Complex legal precedent applications
- Advanced technical issues requiring specialist review
- Policy coverage disputes requiring legal interpretation
- Jurisdictional compliance questions

TIER 3 ANALYSIS:
For each potential finding:
- Finding Type: [Hidden damage/Legal precedent/Technical specialist]
- Investigation Required: [Additional inspections/expert reports needed]
- Specialist Type: [Engineer/Attorney/Industry expert required]
- Recommended Action: [Further investigation/expert consultation]
- Success Probability: 30-60% (pending further investigation)

CONFIDENCE SCORING PROTOCOL:

EVIDENCE STRENGTH ASSESSMENT:
- Documentary Evidence: Photos, reports, measurements with sources
- Regulatory Citations: Specific code sections, standards, regulations
- Mathematical Proof: Calculations, quantity verifications, rate confirmations
- Contradiction Documentation: Direct conflicts between carrier materials

IMPACT ASSESSMENT:
- Financial Impact: Dollar amount of finding
- Legal Impact: Violation severity and potential penalties
- Strategic Impact: Effect on overall case strength
- Timeline Impact: Urgency of response required

FINDINGS INTEGRATION:
Extract and analyze findings from all previous stages:

FROM CODE_LOOKUP STAGE:
- Code violation findings → Tier 1 (if clear) or Tier 2 (if interpretive)
- LKQ violations → Tier 1 (clear mismatch) or Tier 2 (arguable)
- PPA violations → Tier 1 (calculated deadlines exceeded)
- DOI bulletin violations → Tier 2 (regulatory interpretation)

FROM REPORT_ANALYSIS STAGE:
- Photo caption contradictions → Tier 1 (direct conflicts)
- Engineering report conflicts → Tier 2 (expert interpretation needed)
- Measurement discrepancies → Tier 1 (mathematical proof) or Tier 2 (methodology)

FROM SCOPING_LOGIC STAGE:
- Missing code-required items → Tier 1 (clear requirements)
- Industry standard deviations → Tier 2 (interpretation required)
- Aesthetic/matching disputes → Tier 2 (subjective elements)

FROM ESTIMATE STAGE:
- Mathematical errors → Tier 1 (calculation proof)
- Parent-child item issues → Tier 2 (Xactimate interpretation)
- Labor minimum violations → Tier 2 (trade practice standards)

ACTION RECOMMENDATIONS BY TIER:

TIER 1 ACTION PLAN:
- Immediate Response: Draft demand letter within 5 business days
- Documentation Package: Compile all supporting evidence
- Legal Strategy: Pursue formal complaint if carrier doesn't respond
- Timeline: 30-day resolution target

TIER 2 ACTION PLAN:
- Expert Review: Engage appropriate specialist for opinion
- Additional Documentation: Gather supporting industry standards
- Strategic Assessment: Evaluate cost-benefit of pursuit
- Timeline: 60-day investigation and response period

TIER 3 ACTION PLAN:
- Investigation Phase: Conduct additional inspections/testing
- Specialist Engagement: Retain appropriate expert witnesses
- Legal Consultation: Engage coverage counsel for complex issues
- Timeline: 90+ day investigation period

FINAL OUTPUT FORMAT:

## CONFIDENCE SCORING SUMMARY

### TIER 1 - HIGH CONFIDENCE FINDINGS (Immediate Action)
[List all Tier 1 findings with evidence and action plans]

### TIER 2 - MEDIUM CONFIDENCE FINDINGS (Human Review)
[List all Tier 2 findings with interpretation needs and recommendations]

### TIER 3 - POTENTIAL FINDINGS (Expert Escalation)
[List all Tier 3 findings with investigation requirements]

### OVERALL CASE ASSESSMENT
- Total Financial Impact: [Sum of all findings]
- Strongest Arguments: [Top 3 Tier 1 findings]
- Key Vulnerabilities: [Potential carrier defenses]
- Recommended Strategy: [Immediate, medium-term, and long-term actions]
- Success Probability: [Overall case strength assessment]

### ESCALATION MATRIX
| Finding Category | Internal Expertise | External Specialist | Legal Counsel |
|------------------|-------------------|-------------------|---------------|
| [Category] | [Required/Optional] | [Required/Optional] | [Required/Optional] |

Use this confidence framework to prioritize actions and allocate resources effectively based on evidence strength and success probability.
"""
    }

In [8]:
import json
import requests
import os
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths,pricing_path):
    prompts = build_prompts()

    # Assign files
    # 1) Carrier: only the first file
    carrier_parts = [make_part(file_paths[0])]

    # 2) Evidence: file_paths[0] plus file_paths[2:]
    evidence_paths = [file_paths[0]] + file_paths[2:]
    evidence_parts = [make_part(path) for path in evidence_paths]

    # 3) Policy: again, just the first file (if that’s what you meant)
    policy_parts = [make_part(path) for path in file_paths[1:2]]


    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        # run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    # scoping_context = (
    #     f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
    #     f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    # )
#     label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
#     save_output(label, scoping_output)
#     context["SCOPING_LOGIC"] = scoping_output

#     pricing_context = (
#         f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
#         f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
#         f"--- SCOPING LOGIC ---n{context['SCOPING_LOGIC']}"
#     )
    
#     label, pricing_output = await run_block("PRICING_LOGIC", prompts["PRICING_LOGIC"] + "nn" + pricing_context,policy_parts)
#     save_output(label, pricing_output)
#     filepath = os.path.join(pricing_path,"output_pricing_logic.txt")
#     print("Pricing output saved to:", filepath)
#     scope_data = load_llm_scopes_from_file(filepath)
#     if scope_data:
#     # Create estimate
#         valid_scopes = [
#         item for item in scope_data
#         if not item["scope_id"].startswith("MISSING_")
#         ]

# # Call your estimate creation function
#         estimate_response = create_estimate_from_llm_output(valid_scopes, zipcode="80210", estimate_type=2)
#         # print("Estimate response:", estimate_response)
#         # Save estimate_response to file in pricing_path directory
#         estimate_filepath = os.path.join(pricing_path, "estimate_response.json")
#         with open(estimate_filepath, "w") as f:
#             f.write(json.dumps(estimate_response, indent=2))
#         print("Estimate response saved to:", estimate_filepath)
        
    
    



# # Use that structured version in the context
#     estimate_context = (
#     f"--- CODE MANDATES ---\n{context['CODE_LOOKUP']}\n\n"
#     f"--- DAMAGE FINDINGS ---\n{context['REPORT_ANALYSIS']}\n\n" 
#     f"--- SCOPING LOGIC ---\n{context['SCOPING_LOGIC']}\n\n"   
#     f"--- PRICING LOGIC ---\n{pricing_output}\n\n"
#     f"--- SELECTED SCOPES ---\n{estimate_response}\n"
#     )   

#         # f"--- POLICY LOGIC ---n{context['POLICY_LOGIC_output']}nn"
    
#     label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
#     save_output(label, estimate_output)

#     # Stage 4: Rebuttal
#     # Stage 4: Rebuttal (pass carrier file for comparison)
#     label, rebuttal_output = await run_block(
#     "REBUTTAL",
#     prompts["REBUTTAL"] + "nn" + estimate_output,
#     file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
#     )
#     save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        # "report_analysis": context["REPORT_ANALYSIS"],
        # "scoping_logic": context["SCOPING_LOGIC"],
        # "estimate_output": estimate_output,
        # "rebuttal_output": rebuttal_output,
    }


In [10]:
output_dir = f"outputs/Aug10/2500074/run2_gemini_split"
def save_output(label: str, content: str):
   
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",
    # "scopes_for_llm.txt",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf",
],output_dir)

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (1850 chars)
📝 Saved: outputs/Aug10/2500074/run2_gemini_split/output_code_lookup.txt


In [14]:
def save_output(label: str, content: str):
    output_dir = f"outputs/SPLIT_CASE2/RUN3"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_3A97D8FE-84D6-4BFA-91AD-39F43E790DC5.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_093D1853-BBE9-47A7-86EF-1769895C5FBE.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_72CE907A-5EDA-42FC-A9C2-3D4245CC834E.jpeg",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002_Expert_ReportsPhotosDiagrams_From_Fadner, Derek_Messenger_creation_459D33C9-7EA8-451F-A4EE-1C09493F9BEA.jpeg",
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (19698 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (12956 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_code_lookup.txt
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12493 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (13110 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10274 chars)
📝 Saved: outputs/SPLIT_CASE2/RUN3/output_rebuttal.txt
